In [ ]:
import torch
import random
import numpy as np
import pandas as pd
from IPython.display import display
from metrics import compute_pointwise_metrics, precision_recall_thresholds, compute_pr_auc, compute_f1_score
from train_ae import err_fn, build_dataloaders
from train_aae import fuse_scores, train_fusion_model, compute_scores as compute_aae_scores, load_checkpoint, build_dataloaders as build_aae_fusion_dataloader


## Utility functions

In [2]:
@torch.no_grad()
def compute_reconstruction_error_scores(model, dataloader, recon_top_k):
    recon_scores = []
    labels = []

    model.eval()
    for x, y in dataloader:
        recon = model(x)
        
        if isinstance(recon, tuple):
            recon = recon[0]

        # reconstruction error score
        recon_score = (recon - x).pow(2).topk(recon_top_k, dim=1).values.mean(dim=1)

        recon_scores.append(recon_score.cpu())
        labels.append(y.cpu())

    recon_scores = torch.cat(recon_scores)
    labels = torch.cat(labels)

    return recon_scores, labels

def evaluate(model, test_loader, threshold, top_k):
    model.eval()
    all_preds, all_labels, all_errors = [], [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.to(DEVICE)            
            recon = model(x_batch)

            if isinstance(recon, tuple):
                recon = recon[0]
            
            err = err_fn(recon, x_batch, top_k)
            preds = (err > threshold).int()
            
            all_preds.append(preds.cpu())
            all_labels.append(y_batch.cpu())
            all_errors.append(err.cpu())
            
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    all_errors = torch.cat(all_errors)

    return all_preds, all_labels, all_errors

def create_and_display_metrics_df(metrics_list, sort_by=None, ascending=False):
    # Create the DataFrame from the list of dictionaries

    if isinstance(metrics_list, dict):
        metrics_list = [metrics_list]

    df = pd.DataFrame(metrics_list)
    
    # Sort the DataFrame if a sort column is provided and exists
    if sort_by and sort_by in df.columns:
        df = df.sort_values(by=sort_by, ascending=ascending).reset_index(drop=True)
        
    display(df)
    
    return df

# Configuration

In [3]:
# Global Configurations
DATA_DIR = "./data"
SEED = 123
BATCH_SIZE = 1024
LATENT_DIM = 4
LR = 1e-3
LR_AE = 1e-3
LR_D = 5e-5
LR_G = 1e-4
PRETRAIN_EPOCHS = 25 
EPOCHS = 50 
QUANTILE = 0.85
TOP_K = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_FEATURES_RAW = np.load(DATA_DIR + "/KukaNormal.npy").shape[1]


In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Loading Datasets

In [5]:
train_dataset, train_loader, val_loader, test_loader, _ = build_dataloaders(DATA_DIR, BATCH_SIZE, seed=SEED)
x, y = train_dataset[0]
input_dim = x.shape[0]

print(f"Train dataset length {len(train_dataset)}")
print(f"Val dataset length {len(val_loader.dataset)}")
print(f"Test dataset length {len(test_loader.dataset)}")

/Users/marco/Documents/Università/Didattica/Magistrale/Machine Learning in Application/Project/industrial-robot-anomaly-detection/venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/marco/Documents/Università/Didattica/Magistrale/Machine Learning in Application/Project/industrial-robot-anomaly-detection/venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Train dataset length 163654
Val dataset length 55837
Test dataset length 55839


# 1. Base Autoencoder (baseline)

## 1.1 Top K selection

In [6]:
ae = AE(input_dim, LATENT_DIM).to(DEVICE)
ae.load_state_dict(torch.load("ae_best.pth", map_location=DEVICE))

top_ks_to_test = [1, 2, 3, 5, 10, 15, 20, 25, 30, 40, input_dim]
results = []

for k in top_ks_to_test:

    print("-" * 60)
    print(f"Testing AE ->  top_k: {k}")
    
    scores, labels = compute_reconstruction_error_scores(ae, val_loader, k)
    auc = compute_pr_auc(scores, labels)
    
    results.append({
        "top_k": k,
        "pr_auc": auc
    })
    
df = create_and_display_metrics_df(results)
best_ae_top_k = df.loc[df['pr_auc'].idxmax(), 'top_k'].item()
best_ae_top_k

------------------------------------------------------------
Testing AE ->  top_k: 1
------------------------------------------------------------
Testing AE ->  top_k: 2
------------------------------------------------------------
Testing AE ->  top_k: 3
------------------------------------------------------------
Testing AE ->  top_k: 5
------------------------------------------------------------
Testing AE ->  top_k: 10
------------------------------------------------------------
Testing AE ->  top_k: 15
------------------------------------------------------------
Testing AE ->  top_k: 20
------------------------------------------------------------
Testing AE ->  top_k: 25
------------------------------------------------------------
Testing AE ->  top_k: 30
------------------------------------------------------------
Testing AE ->  top_k: 40
------------------------------------------------------------
Testing AE ->  top_k: 82


,top_k,pr_auc
0,1,0.848639
1,2,0.849545
2,3,0.845467
3,5,0.843348
4,10,0.845219
5,15,0.846069
6,20,0.845630
7,25,0.844577
8,30,0.843387
9,40,0.841573


2

## 1.2 Threshold tuning 

In [7]:
scores, labels = compute_reconstruction_error_scores(ae, val_loader, best_ae_top_k)
precisions, recalls, thresolds = precision_recall_thresholds(scores, labels)

f1_scores = np.divide(
    2 * precisions * recalls,
    precisions + recalls,
    out=np.zeros_like(precisions),
    where=(precisions + recalls) != 0
)

best_idx = np.argmax(f1_scores)
best_ae_threshold = thresolds[best_idx].item()
best_ae_threshold
    

0.087016761302948

## 1.3 Testing AE best model

In [8]:
all_preds, all_labels, all_errors = evaluate(ae, test_loader, best_ae_threshold, best_ae_top_k)
_ = create_and_display_metrics_df(compute_pointwise_metrics(all_labels, all_preds, all_errors, 2.0))

,precision,recall,f1,f2,accuracy,balanced_accuracy,pr_auc,average_precision,n_samples,n_normal,n_anomaly
0,0.623921,0.859502,0.723005,0.799153,0.755046,0.776344,0.848445,0.848461,55839,35070,20769


# 2. Base Adversarial Autoencoder

## 2.1 Top K selection

In [9]:
aae, outlier_estimator, fusion = load_checkpoint("weights/aae/aae_best_top_k1.pth", device=DEVICE)

top_ks_to_test = [1, 2, 3, 5, 10, 15, 20, 25, 30, 40, input_dim]
results = []

for k in top_ks_to_test:

    print("-" * 60)
    print(f"Testing AAE ->  top_k: {k}")
    
    scores, labels = compute_reconstruction_error_scores(aae, val_loader, k)
    auc = compute_pr_auc(scores, labels)
    
    results.append({
        "top_k": k,
        "pr_auc": auc
    })
    
df = create_and_display_metrics_df(results)
best_aae_top_k = df.loc[df['pr_auc'].idxmax(), 'top_k'].item()
best_aae_top_k

------------------------------------------------------------
Testing AAE ->  top_k: 1
------------------------------------------------------------
Testing AAE ->  top_k: 2
------------------------------------------------------------
Testing AAE ->  top_k: 3
------------------------------------------------------------
Testing AAE ->  top_k: 5
------------------------------------------------------------
Testing AAE ->  top_k: 10
------------------------------------------------------------
Testing AAE ->  top_k: 15
------------------------------------------------------------
Testing AAE ->  top_k: 20
------------------------------------------------------------
Testing AAE ->  top_k: 25
------------------------------------------------------------
Testing AAE ->  top_k: 30
------------------------------------------------------------
Testing AAE ->  top_k: 40
------------------------------------------------------------
Testing AAE ->  top_k: 82


,top_k,pr_auc
0,1,0.917871
1,2,0.917995
2,3,0.914965
3,5,0.911366
4,10,0.912546
5,15,0.913283
6,20,0.912701
7,25,0.911468
8,30,0.910029
9,40,0.907744


2

## 2.2 Threshold tuning 

In [10]:
scores, labels = compute_reconstruction_error_scores(aae, val_loader, best_aae_top_k)
precisions, recalls, thresolds = precision_recall_thresholds(scores, labels)

f1_scores = np.divide(
    2 * precisions * recalls,
    precisions + recalls,
    out=np.zeros_like(precisions),
    where=(precisions + recalls) != 0
)

best_idx = np.argmax(f1_scores)
best_aae_threshold = thresolds[best_idx].item()
best_aae_threshold
    

0.22363901138305664

## 2.3 Testing AAE best model

In [11]:
all_preds, all_labels, all_errors = evaluate(aae, test_loader, best_aae_threshold, best_aae_top_k)
_ = create_and_display_metrics_df(compute_pointwise_metrics(all_labels, all_preds, all_errors, 2.0))

,precision,recall,f1,f2,accuracy,balanced_accuracy,pr_auc,average_precision,n_samples,n_normal,n_anomaly
0,0.958676,0.730512,0.829184,0.767022,0.888053,0.855932,0.920335,0.920341,55839,35070,20769


# 3 Adversarial Autoencoder + Score Fusion

## 3.0 Further dataset splitting

In [12]:
train_loader, train_fuse_loader, val_loader, test_loader, _ = build_aae_fusion_dataloader(DATA_DIR, BATCH_SIZE, seed=SEED)

print(f"Train dataset length {len(train_dataset)}")
print(f"Fusion training dataset length {len(train_fuse_loader.dataset)}")
print(f"Val dataset length {len(val_loader.dataset)}")
print(f"Test dataset length {len(test_loader.dataset)}")

/Users/marco/Documents/Università/Didattica/Magistrale/Machine Learning in Application/Project/industrial-robot-anomaly-detection/venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/marco/Documents/Università/Didattica/Magistrale/Machine Learning in Application/Project/industrial-robot-anomaly-detection/venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Train dataset length 163654
Fusion training dataset length 27918
Val dataset length 27919
Test dataset length 55839


## 3.1 Top K selection

In [13]:
top_ks_to_test = [1, 2, 3, 5, 10, 15, 20, 25, 30, 40, input_dim]
results = []

for k in top_ks_to_test:

    print("-" * 60)
    print(f"Testing AAE + fusion -> top_k: {k}")
    
    recon_scores, outlier_scores, latent_scores, labels = compute_aae_scores(aae, train_fuse_loader, k, outlier_estimator)
    fusion_model = train_fusion_model(labels, recon_scores, outlier_scores, latent_scores)
    
    recon_scores, outlier_scores, latent_scores, labels = compute_aae_scores(aae, val_loader, k, outlier_estimator)
    scores = fuse_scores(fusion_model, recon_scores, outlier_scores, latent_scores)
    
    t = 0.0
    preds = (scores > t).int()
    f1 = compute_f1_score(preds, labels)
    
    results.append({
        "top_k": k,
        "f1_score": f1
    })
    
df = create_and_display_metrics_df(results)
best_aae_fusion_top_k = df.loc[df['f1_score'].idxmax(), 'top_k'].item()
best_aae_fusion_top_k

------------------------------------------------------------
Testing AAE + fusion -> top_k: 1
------------------------------------------------------------
Testing AAE + fusion -> top_k: 2
------------------------------------------------------------
Testing AAE + fusion -> top_k: 3
------------------------------------------------------------
Testing AAE + fusion -> top_k: 5
------------------------------------------------------------
Testing AAE + fusion -> top_k: 10
------------------------------------------------------------
Testing AAE + fusion -> top_k: 15
------------------------------------------------------------
Testing AAE + fusion -> top_k: 20
------------------------------------------------------------
Testing AAE + fusion -> top_k: 25
------------------------------------------------------------
Testing AAE + fusion -> top_k: 30
------------------------------------------------------------
Testing AAE + fusion -> top_k: 40
------------------------------------------------------

,top_k,f1_score
0,1,0.945558
1,2,0.941176
2,3,0.936527
3,5,0.931860
4,10,0.931124
5,15,0.931934
6,20,0.931702
7,25,0.932605
8,30,0.932652
9,40,0.934903


1